# Random Forest - Phan loai binh luan tieu cuc Tiki

**Muc tieu**: Train Random Forest tren dataset augmented, tap trung kiem tra overfitting va xu ly imbalance.

**Dataset**: `comments_merged_single_label_augmented.csv` (26,110 mau, 4 lop)

**Imbalance**: Quality 42.55% vs Service/Shipping/Packing ~19% moi lop

## 1. Import & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, learning_curve, cross_val_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)
import joblib
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE = 0.2
LABEL_COLS = ['Quality', 'Service', 'Shipping', 'Packing']
LABEL_MAP = {0: 'Quality', 1: 'Service', 2: 'Shipping', 3: 'Packing'}

## 2. Load Data & Tao Label

In [ ]:
# Kaggle path - thay doi neu chay local
DATA_PATH = '/kaggle/input/datasets/lcminhc/tiki-comments-single-label-augmented/comments_merged_single_label_augmented.csv'
# LOCAL: DATA_PATH = '../data/Commentsauxuly/comments_merged_single_label_augmented.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
df['label'] = df[LABEL_COLS].values.argmax(axis=1)
df['label_name'] = df['label'].map(LABEL_MAP)

print('Label distribution:')
print(df['label_name'].value_counts())
print(f'\nTotal samples: {len(df)}')
print(f'Null content: {df["content"].isnull().sum()}')

## 3. EDA - Phan tich du lieu

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart phan bo label
label_counts = df['label_name'].value_counts()
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
ax = label_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Phan bo label', fontsize=14)
axes[0].set_ylabel('So luong')
axes[0].set_xticklabels(label_counts.index, rotation=0)
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 100, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

# Imbalance ratio
imbalance_ratio = label_counts.max() / label_counts.min()
axes[0].text(0.5, 0.95, f'Imbalance ratio: {imbalance_ratio:.2f}x',
             transform=axes[0].transAxes, ha='center', fontsize=11,
             bbox=dict(boxstyle='round', facecolor='wheat'))

# Histogram do dai text
df['text_len'] = df['content'].astype(str).apply(len)
for label_name, color in zip(LABEL_MAP.values(), colors):
    subset = df[df['label_name'] == label_name]['text_len']
    axes[1].hist(subset, bins=50, alpha=0.5, label=label_name, color=color)
axes[1].set_title('Do dai text theo label', fontsize=14)
axes[1].set_xlabel('So ky tu')
axes[1].set_ylabel('So luong')
axes[1].legend()
axes[1].set_xlim(0, 500)

# Pie chart
axes[2].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[2].set_title('Ty le label', fontsize=14)

plt.tight_layout()
plt.show()

print('\nThong ke do dai text:')
print(df.groupby('label_name')['text_len'].describe().round(1))

## 4. Train/Test Split (Stratified)

In [ ]:
X = df['content'].astype(str)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')
print(f'\nTrain label distribution:')
print(y_train.map(LABEL_MAP).value_counts())
print(f'\nTest label distribution:')
print(y_test.map(LABEL_MAP).value_counts())

## 5. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

## 6. Baseline RF (khong xu ly imbalance)

Train RF voi default params de lam baseline so sanh.

In [ ]:
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_baseline.fit(X_train_tfidf, y_train)

train_acc_baseline = rf_baseline.score(X_train_tfidf, y_train)
test_acc_baseline = rf_baseline.score(X_test_tfidf, y_test)
y_pred_baseline = rf_baseline.predict(X_test_tfidf)
f1_w_baseline = f1_score(y_test, y_pred_baseline, average='weighted')
f1_m_baseline = f1_score(y_test, y_pred_baseline, average='macro')

print('=== BASELINE RF (no class_weight) ===')
print(f'Train Accuracy: {train_acc_baseline:.5f}')
print(f'Test Accuracy:  {test_acc_baseline:.5f}')
print(f'Gap (Train-Test): {train_acc_baseline - test_acc_baseline:.5f}')
print(f'F1 Weighted: {f1_w_baseline:.5f}')
print(f'F1 Macro:    {f1_m_baseline:.5f}')
print()

if train_acc_baseline - test_acc_baseline > 0.05:
    print('>>> CANH BAO: Gap > 5% => CO DAU HIEU OVERFITTING!')
else:
    print('>>> Gap < 5% => Chua co dau hieu overfit ro rang.')

print('\n--- Classification Report (Test) ---')
print(classification_report(y_test, y_pred_baseline, target_names=list(LABEL_MAP.values())))

## 7. RF voi class_weight='balanced_subsample'

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_balanced.fit(X_train_tfidf, y_train)

train_acc_balanced = rf_balanced.score(X_train_tfidf, y_train)
test_acc_balanced = rf_balanced.score(X_test_tfidf, y_test)
y_pred_balanced = rf_balanced.predict(X_test_tfidf)
f1_w_balanced = f1_score(y_test, y_pred_balanced, average='weighted')
f1_m_balanced = f1_score(y_test, y_pred_balanced, average='macro')

print('=== RF voi class_weight=balanced_subsample ===')
print(f'Train Accuracy: {train_acc_balanced:.5f}')
print(f'Test Accuracy:  {test_acc_balanced:.5f}')
print(f'Gap (Train-Test): {train_acc_balanced - test_acc_balanced:.5f}')
print(f'F1 Weighted: {f1_w_balanced:.5f}')
print(f'F1 Macro:    {f1_m_balanced:.5f}')
print()

if train_acc_balanced - test_acc_balanced > 0.05:
    print('>>> CANH BAO: Gap > 5% => CO DAU HIEU OVERFITTING!')
else:
    print('>>> Gap < 5% => Chua co dau hieu overfit ro rang.')

print('\n--- Classification Report (Test) ---')
print(classification_report(y_test, y_pred_balanced, target_names=list(LABEL_MAP.values())))

## 8. Overfitting Detection: Learning Curve

Plot train score vs validation score khi tang dan training size.
- **Overfit**: train score cao, validation score thap, gap khong giam
- **Good fit**: 2 duong hoi tu

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    X_train_tfidf, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_weighted',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
plt.plot(train_sizes, val_mean, 'o-', color='red', label='Validation score')
plt.xlabel('Training set size', fontsize=12)
plt.ylabel('F1 Weighted Score', fontsize=12)
plt.title('Learning Curve - Random Forest (balanced_subsample)', fontsize=14)
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 1.02)
plt.tight_layout()
plt.show()

print(f'Final gap (train - val): {train_mean[-1] - val_mean[-1]:.4f}')
print(f'Validation score at 100% data: {val_mean[-1]:.4f} (+/- {val_std[-1]:.4f})')

## 9. Overfitting Detection: Cross-Validation Variance

Neu std > 0.02 giua cac fold => model khong on dinh, co the overfit.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_f1_weighted = cross_val_score(
    RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    X_train_tfidf, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1
)

cv_f1_macro = cross_val_score(
    RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    X_train_tfidf, y_train, cv=cv, scoring='f1_macro', n_jobs=-1
)

print('=== 5-Fold Cross-Validation ===')
print(f'F1 Weighted per fold: {cv_f1_weighted}')
print(f'Mean: {cv_f1_weighted.mean():.4f} | Std: {cv_f1_weighted.std():.4f}')
print()
print(f'F1 Macro per fold: {cv_f1_macro}')
print(f'Mean: {cv_f1_macro.mean():.4f} | Std: {cv_f1_macro.std():.4f}')
print()

if cv_f1_weighted.std() > 0.02:
    print('>>> CANH BAO: Std > 0.02 => Model KHONG ON DINH, co the overfit!')
else:
    print('>>> Std < 0.02 => Model on dinh giua cac fold.')

## 10. Overfitting Detection: Train vs Test per Class

So sanh F1 tren train set va test set cho tung lop.
- Neu train F1 ~ 1.0 nhung test F1 < 0.9 => Overfit
- Dac biet kiem tra Service/Shipping/Packing (lop thieu so)

In [ ]:
y_train_pred = rf_balanced.predict(X_train_tfidf)
y_test_pred = rf_balanced.predict(X_test_tfidf)

print('=== TRAIN SET ===')
report_train = classification_report(
    y_train, y_train_pred, target_names=list(LABEL_MAP.values()), output_dict=True
)
print(classification_report(y_train, y_train_pred, target_names=list(LABEL_MAP.values())))

print('\n=== TEST SET ===')
report_test = classification_report(
    y_test, y_test_pred, target_names=list(LABEL_MAP.values()), output_dict=True
)
print(classification_report(y_test, y_test_pred, target_names=list(LABEL_MAP.values())))

# So sanh F1 per class
print('\n=== SO SANH F1 PER CLASS (Train vs Test) ===')
print(f'{"Class":<12} {"Train F1":>10} {"Test F1":>10} {"Gap":>10} {"Overfit?":>10}')
print('-' * 55)
for label_name in LABEL_MAP.values():
    train_f1 = report_train[label_name]['f1-score']
    test_f1 = report_test[label_name]['f1-score']
    gap = train_f1 - test_f1
    overfit = 'YES' if (train_f1 > 0.99 and test_f1 < 0.90) else 'no'
    print(f'{label_name:<12} {train_f1:>10.4f} {test_f1:>10.4f} {gap:>10.4f} {overfit:>10}')

## 11. Hyperparameter Tuning (giam overfit)

GridSearchCV voi cac tham so gioi han do phuc tap cua model.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [20, 30, 50, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced_subsample']
}

grid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid,
    cv=grid_cv,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train_tfidf, y_train)

print(f'\nBest params: {grid_search.best_params_}')
print(f'Best CV F1 weighted: {grid_search.best_score_:.5f}')

In [ ]:
# Kiem tra overfit cua best model tu GridSearch
best_rf = grid_search.best_estimator_

train_acc_tuned = best_rf.score(X_train_tfidf, y_train)
test_acc_tuned = best_rf.score(X_test_tfidf, y_test)
y_pred_tuned = best_rf.predict(X_test_tfidf)
f1_w_tuned = f1_score(y_test, y_pred_tuned, average='weighted')
f1_m_tuned = f1_score(y_test, y_pred_tuned, average='macro')

print('=== TUNED RF (Best GridSearchCV) ===')
print(f'Train Accuracy: {train_acc_tuned:.5f}')
print(f'Test Accuracy:  {test_acc_tuned:.5f}')
print(f'Gap (Train-Test): {train_acc_tuned - test_acc_tuned:.5f}')
print(f'F1 Weighted: {f1_w_tuned:.5f}')
print(f'F1 Macro:    {f1_m_tuned:.5f}')

print('\n--- Classification Report (Test) ---')
print(classification_report(y_test, y_pred_tuned, target_names=list(LABEL_MAP.values())))

## 12. Evaluation - Confusion Matrix & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_tuned)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

labels_annot = np.array([
    [f'{val}\n({pct:.1f}%)' for val, pct in zip(row_val, row_pct)]
    for row_val, row_pct in zip(cm, cm_pct)
])

sns.heatmap(cm, annot=labels_annot, fmt='', cmap='Blues',
            xticklabels=list(LABEL_MAP.values()),
            yticklabels=list(LABEL_MAP.values()),
            ax=axes[0])
axes[0].set_title('Confusion Matrix (Tuned RF)', fontsize=14)
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature Importance - Top 20
feature_names = tfidf.get_feature_names_out()
importances = best_rf.feature_importances_
top_idx = np.argsort(importances)[-20:]
axes[1].barh(range(20), importances[top_idx], color='steelblue')
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(feature_names[top_idx])
axes[1].set_title('Top 20 Feature Importance', fontsize=14)
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 13. Bang so sanh 3 variants

In [ ]:
results = pd.DataFrame({
    'Model': ['Baseline (no weight)', 'balanced_subsample', 'Tuned (GridSearchCV)'],
    'Train Acc': [train_acc_baseline, train_acc_balanced, train_acc_tuned],
    'Test Acc': [test_acc_baseline, test_acc_balanced, test_acc_tuned],
    'Gap': [
        train_acc_baseline - test_acc_baseline,
        train_acc_balanced - test_acc_balanced,
        train_acc_tuned - test_acc_tuned
    ],
    'F1 Weighted': [f1_w_baseline, f1_w_balanced, f1_w_tuned],
    'F1 Macro': [f1_m_baseline, f1_m_balanced, f1_m_tuned]
})

print('=== BANG SO SANH 3 VARIANTS ===')
print(results.to_string(index=False, float_format='%.5f'))
print()

best_model_name = results.loc[results['F1 Weighted'].idxmax(), 'Model']
print(f'>>> Model tot nhat (F1 Weighted): {best_model_name}')

# Highlight overfit
for _, row in results.iterrows():
    if row['Gap'] > 0.05:
        print(f'>>> {row["Model"]}: Gap = {row["Gap"]:.4f} > 0.05 => OVERFIT')

## 14. Ket luan

Dua tren cac ket qua tren, danh gia:
1. Dataset co overfit hay khong?
2. Augmented data co gay van de khong?
3. Recommendations cho cac model tiep theo

In [ ]:
print('=' * 60)
print('KET LUAN')
print('=' * 60)

gap_tuned = train_acc_tuned - test_acc_tuned

print(f'\n1. OVERFIT CHECK:')
if gap_tuned > 0.10:
    print(f'   - Gap = {gap_tuned:.4f} => OVERFIT NGHIEM TRONG')
    print(f'   - Can xem xet lai data augmentation hoac giam max_features/max_depth')
elif gap_tuned > 0.05:
    print(f'   - Gap = {gap_tuned:.4f} => OVERFIT NHE')
    print(f'   - Co the chap nhan, nhung nen regularize them')
else:
    print(f'   - Gap = {gap_tuned:.4f} => KHONG OVERFIT')
    print(f'   - Model generalize tot')

print(f'\n2. IMBALANCE HANDLING:')
f1_diff = f1_w_tuned - f1_w_baseline
print(f'   - F1 improvement voi balanced_subsample: {f1_diff:+.4f}')
if f1_diff > 0:
    print(f'   - class_weight giup cai thien => Imbalance LA van de')
else:
    print(f'   - class_weight khong cai thien => Imbalance KHONG phai van de lon')

print(f'\n3. BEST MODEL:')
print(f'   - Test Accuracy: {test_acc_tuned:.5f}')
print(f'   - F1 Weighted: {f1_w_tuned:.5f}')
print(f'   - F1 Macro: {f1_m_tuned:.5f}')
print(f'   - Params: {grid_search.best_params_}')

## 15. Save Model

In [ ]:
joblib.dump(best_rf, 'rf_classifier_tuned.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print('Saved: rf_classifier_tuned.pkl')
print('Saved: tfidf_vectorizer.pkl')